In [0]:
from pyspark.sql import functions as F, types as T

rows_customers = [
    (1,  "Asha",  "IN", True),
    (2,  "Bob",   "US", False),
    (3,  "Chen",  "CN", True),
    (4,  "Diana", "US", None),
    (None, "Ghost","UK", False),     # NULL key to demo null join behavior
]

rows_orders = [
    (101, 1,   120.0, "IN"),
    (102, 1,    80.0, "IN"),
    (103, 2,    50.0, "US"),
    (104, 5,    30.0, "DE"),         # no matching customer_id
    (105, 3,   200.0, "CN"),
    (106, None, 15.0, "UK"),         # NULL key won’t match
    (107, 3,    40.0, "CN"),
    (108, 2,    75.0, "US"),
]
schema_customers = T.StructType([
    T.StructField("id", T.IntegerType(), True),
    T.StructField("name", T.StringType(), False),
    T.StructField("country", T.StringType(), False),
    T.StructField("is_vip", T.BooleanType())
])
schema_orders = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount", T.DoubleType(), False),
    T.StructField("country", T.StringType(), False)
])
customers = spark.createDataFrame(rows_customers, schema_customers)
orders = spark.createDataFrame(rows_orders, schema_orders)

display(customers)
display(orders)

In [0]:
join_table = customers.join(
    orders,
    customers.id == orders.customer_id,
    "inner"
)

display(join_table)



In [0]:
o=orders.alias("o")
c=customers.alias("c")
o.show()

In [0]:
from pyspark.sql import functions as F

inner_clean_table = (
    o.join(
        c,
        o.customer_id == c.id,
        "inner"
    )
    .select(
        "order_id",
        "customer_id",
        "amount",
        F.col("o.country").alias("ship_country"),
        "name",
        F.col("c.country").alias("cust_country"),
        "is_vip"
    )
)

display(inner_clean_table)
